# Resolución de la tarea — `z402_Feature_Engineering_en_SQL`

**Consigna original:** `monday/z402_Feature_Engineering_en_SQL.ipynb` (Alejandro Bolaños)

> **TAREA:** Escriba una macro para hacer un ratio de dos variables que sea seguro,
> donde no solo hay campos con null, también esta el problema de la división por cero.
> Como es costumbre comparta su solución por este canal. Lea
> https://duckdb.org/docs/sql/functions/numeric.html para referencias de funciones que
> puede usar.

Sigue el mismo esquema del notebook original: `jupysql` sobre `duckdb`, todo en `%%sql`.

## Entorno

Igual que el original. Si falta algo: `pip install jupysql duckdb-engine`.

In [1]:
import duckdb
import pandas as pd

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

%sql duckdb:///

Única diferencia con el original: la ruta. Los datos están en el repo, no en Drive.

In [2]:
dataset_path = '../../monday/'
dataset_file = 'competencia_01.csv'

In [3]:
%%sql
create or replace table competencia_01 as
select
    *
from read_csv_auto("{{dataset_path + dataset_file}}")

,Success


## De dónde partimos

El notebook original llega hasta acá: una macro que suma dos campos sin que un `null`
se coma el resultado.

In [4]:
%%sql
CREATE OR REPLACE MACRO suma_sin_null(a, b) AS ifnull(a, 0) + ifnull(b, 0);

,Success


---

# La resolución

## Qué se busca

Que un ratio `a / b` no rompa nada corriente abajo. Un ratio tiene **tres formas de
salir mal**, y solo una es la que la consigna nombra primero:

1. **`a` o `b` son `null`** — el resultado es `null` y se propaga
2. **`b = 0`** — división por cero
3. **`a = 0` y `b = 0`** — indeterminado, que no es lo mismo que el caso anterior

Pero la trampa de verdad está un paso más adelante, y es lo primero que hay que mirar.

## Diagnóstico: qué hace DuckDB realmente

Antes de escribir nada, hay que ver qué devuelve el motor. La suposición natural es
que dividir por cero tira error. **No es así.**

In [5]:
%%sql
select
    1 / 0                as entero_sobre_cero
  , 1.0 / 0.0            as float_sobre_cero
  , -1.0 / 0.0           as negativo_sobre_cero
  , 0.0 / 0.0            as cero_sobre_cero
  , isinf(1.0 / 0.0)     as es_infinito
  , isnan(0.0 / 0.0)     as es_nan

,entero_sobre_cero,float_sobre_cero,negativo_sobre_cero,cero_sobre_cero,es_infinito,es_nan
0,inf,inf,-inf,NaN,True,True


### Lo que muestra, y por qué importa

DuckDB **no falla**: devuelve `inf`, `-inf` y `nan` sin decir nada.

Eso es peor que un error. Un error frena la corrida y te obliga a mirar. Un `inf`
sigue de largo:

- Se guarda en el `.csv` como el texto `inf`
- Al releerlo, la columna entera puede quedar tipada como texto
- Si llega a `scikit-learn`, tira `Input contains infinity or a value too large`
- Si llega a un modelo que lo tolera, **el árbol parte por `inf`** y aprende una regla
  que no significa nada

Un `nan` es todavía peor, porque `nan != nan`: se filtra por comparaciones que
deberían atraparlo.

La macro no está para evitar un error. Está para **evitar el silencio**.

## Una trampa: `divide()`

Leyendo la documentación de funciones numéricas que sugiere la consigna, aparece
`divide()` y parece la respuesta: devuelve `null` en vez de `inf`. Pero:

In [6]:
%%sql
select
    divide(1, 0)   as divide_por_cero      -- devuelve null, prometedor
  , divide(7, 2)   as divide_siete_dos     -- ... pero acá está el problema
  , 7 // 2         as doble_barra

,divide_por_cero,divide_siete_dos,doble_barra
0,<NA>,3,3


`divide()` es **división entera** — es el operador `//`. Un ratio de `7` sobre `2` te
da `3`, no `3.5`. Sirve para la división por cero y arruina todo lo demás. No es la
herramienta.

La que sí sirve es **`nullif(b, 0)`**: convierte el cero en `null`, y dividir por
`null` da `null`. Sin infinitos.

In [7]:
%%sql
select
    3 / nullif(0, 0)   as con_nullif
  , 3.0 / 0.0          as sin_nullif

,con_nullif,sin_nullif
0,NaN,inf


## La decisión de diseño: qué hacer con los `null`

Acá hay que elegir, y la elección no es técnica sino del problema.

La tentación es copiar el patrón de `suma_sin_null` y hacer `ifnull(a, 0) / ifnull(b, 0)`.
**En este dataset eso destruye información.**

En el EDA de `z201` salió que el 4,89% de los clientes tiene todo el bloque `Visa_*`
en `null` — no porque falte el dato, sino porque **no tienen tarjeta Visa**. Y entre
ellos la tasa de `BAJA+2` es 3,84% contra 0,70% general: **lift 5,5×**. El `null` ahí
no es un agujero, es un hecho del cliente.

Convertirlo en `0` le dice al modelo "tiene tarjeta con saldo cero", que es otra cosa
completamente distinta y además falsa.

Por eso la macro **propaga el `null`** en vez de taparlo. `null` significa "no sé", y
el modelo tiene que poder distinguir "no sé" de "cero".

## La macro

In [8]:
%%sql
CREATE OR REPLACE MACRO ratio_seguro(a, b) AS
    case
        when a is null or b is null then null   -- falta el dato: no lo invento
        when a = 0 and b = 0        then 0      -- nada sobre nada: sin actividad
        when b = 0                  then null   -- a/0 es indefinido, no infinito
        else a / nullif(b, 0)                   -- red de seguridad
    end;

,Success


### Cómo se lee

Cuatro casos, en orden, y cada rama es una decisión explícita:

| caso | devuelve | por qué |
|---|---|---|
| `a` o `b` es `null` | `null` | el dato falta y eso es información, no se inventa |
| `0 / 0` | `0` | el cliente no tuvo actividad en ninguno de los dos campos |
| `a / 0` con `a ≠ 0` | `null` | matemáticamente indefinido — `inf` sería mentir |
| resto | `a / b` | el `nullif` queda igual, como red por si alguna rama falla |

La distinción entre `0/0` y `a/0` es la que se suele saltear. No son el mismo caso:
en el primero el cliente simplemente no operó, y `0` es una respuesta honesta. En el
segundo hay numerador sin denominador, y ahí no hay número que sirva.

### La prueba: todos los casos límite juntos

In [9]:
%%sql
with casos(descripcion, a, b) as (
    values ('normal',            10.0, 4.0)
         , ('cero sobre algo',    0.0, 4.0)
         , ('algo sobre cero',   10.0, 0.0)
         , ('cero sobre cero',    0.0, 0.0)
         , ('numerador null',    null, 4.0)
         , ('denominador null',  10.0, null)
         , ('ambos null',        null, null)
         , ('negativo',          -8.0, 4.0)
)
select
    descripcion
  , a, b
  , a / b                  as division_cruda
  , ratio_seguro(a, b)     as con_la_macro
from casos

,descripcion,a,b,division_cruda,con_la_macro
0,normal,10.0,4.0,2.5,2.5
1,cero sobre algo,0.0,4.0,0.0,0.0
2,algo sobre cero,10.0,0.0,inf,NaN
3,cero sobre cero,0.0,0.0,NaN,0.0
4,numerador null,NaN,4.0,NaN,NaN
5,denominador null,10.0,NaN,NaN,NaN
6,ambos null,NaN,NaN,NaN,NaN
7,negativo,-8.0,4.0,-2.0,-2.0


La columna `division_cruda` es lo que pasaría sin la macro: `inf`, `-inf` y `nan`
metiéndose en el dataset. La columna de la derecha no tiene ninguno.

## Variante con valor por defecto

A veces sí querés un número en vez de `null` — por ejemplo si el modelo que viene
después no tolera faltantes. En ese caso, que la sustitución sea **explícita y en el
llamado**, no escondida en la macro:

In [10]:
%%sql
CREATE OR REPLACE MACRO ratio_seguro_def(a, b, defecto) AS
    coalesce(ratio_seguro(a, b), defecto);

,Success


In [11]:
%%sql
select
    ratio_seguro(10.0, 0.0)          as sin_defecto
  , ratio_seguro_def(10.0, 0.0, -1)  as con_defecto_menos_uno
  , ratio_seguro_def(null, 4.0, 0)   as null_reemplazado

,sin_defecto,con_defecto_menos_uno,null_reemplazado
0,NaN,-1.0,0.0


---

## Sobre datos reales

Los dos modos de falla no aparecen en el mismo par de variables, así que hacen falta dos
ejemplos.

**El cero real** está en `mcaja_ahorro`: 11.900 clientes de 202104 tienen la caja en cero.
El ratio *cuánto consume con la tarjeta contra lo que tiene en la caja de ahorro* pisa
esa mina de lleno.

**El null** está en el bloque `Visa_*`, que es `null` para el 4,89% que no tiene tarjeta.
Curiosamente `Visa_mlimitecompra` **nunca vale cero** — o hay límite, o no hay tarjeta.
Es un buen recordatorio de que null y cero son cosas distintas también en los datos, no
solo en la macro.

In [12]:
%%sql
select
    count(*)                                                             as clientes
  , sum(mcaja_ahorro = 0)::int                                           as denominador_cero
  , sum(mtarjeta_visa_consumo = 0 and mcaja_ahorro = 0)::int             as cero_sobre_cero
  , sum(mtarjeta_visa_consumo <> 0 and mcaja_ahorro = 0)::int            as seria_infinito
  , sum(Visa_mlimitecompra is null)::int                                 as bloque_visa_null
  , sum(Visa_mlimitecompra = 0)::int                                     as limite_visa_cero
from competencia_01
where foto_mes = 202104

,clientes,denominador_cero,cero_sobre_cero,seria_infinito,bloque_visa_null,limite_visa_cero
0,163284,11900,4930,6970,7987,0


`seria_infinito` son las filas que, con una división cruda, entrarían al dataset como
`inf`. Y `cero_sobre_cero` son las que entrarían como `nan`. Veamos las dos formas lado
a lado, justo sobre las filas problemáticas:

In [13]:
%%sql
select
    mtarjeta_visa_consumo
  , mcaja_ahorro
  , mtarjeta_visa_consumo / mcaja_ahorro                    as consumo_crudo
  , ratio_seguro(mtarjeta_visa_consumo, mcaja_ahorro)       as consumo_seguro
from competencia_01
where foto_mes = 202104
  and mcaja_ahorro = 0
order by mtarjeta_visa_consumo desc
limit 8

,mtarjeta_visa_consumo,mcaja_ahorro,consumo_crudo,consumo_seguro
0,1019184.08,0.0,inf,NaN
1,950490.24,0.0,inf,NaN
2,789771.65,0.0,inf,NaN
3,676985.60,0.0,inf,NaN
4,634644.61,0.0,inf,NaN
5,619429.26,0.0,inf,NaN
6,459663.08,0.0,inf,NaN
7,432288.58,0.0,inf,NaN


## Generar muchos ratios de una

El original cierra mostrando cómo generar el texto de la query con un `for` de Python
en vez de escribir cien líneas a mano. Mismo truco, con la macro adentro.

Elijo pares que tengan sentido junto, no todas las combinaciones posibles:

In [14]:
pares = [
    ("Visa_msaldototal",      "Visa_mlimitecompra",  "visa_utilizacion"),
    ("Master_msaldototal",    "Master_mlimitecompra", "master_utilizacion"),
    ("mtarjeta_visa_consumo", "mcaja_ahorro",        "consumo_sobre_caja"),
    ("mtarjeta_visa_consumo", "mcuentas_saldo",      "consumo_sobre_saldo"),
    ("mpayroll",              "mrentabilidad",       "payroll_sobre_rentabilidad"),
    ("ctrx_quarter",          "cproductos",          "trx_por_producto"),
]

nuevos_features = ""
for a, b, nombre in pares:
    nuevos_features += f"\n  , ratio_seguro({a}, {b}) as {nombre}"
print(nuevos_features)


  , ratio_seguro(Visa_msaldototal, Visa_mlimitecompra) as visa_utilizacion
  , ratio_seguro(Master_msaldototal, Master_mlimitecompra) as master_utilizacion
  , ratio_seguro(mtarjeta_visa_consumo, mcaja_ahorro) as consumo_sobre_caja
  , ratio_seguro(mtarjeta_visa_consumo, mcuentas_saldo) as consumo_sobre_saldo
  , ratio_seguro(mpayroll, mrentabilidad) as payroll_sobre_rentabilidad
  , ratio_seguro(ctrx_quarter, cproductos) as trx_por_producto


In [15]:
%%sql
select
    numero_de_cliente
  , foto_mes
  , ratio_seguro(Visa_msaldototal, Visa_mlimitecompra) as visa_utilizacion
  , ratio_seguro(Master_msaldototal, Master_mlimitecompra) as master_utilizacion
  , ratio_seguro(mtarjeta_visa_consumo, mcaja_ahorro) as consumo_sobre_caja
  , ratio_seguro(mtarjeta_visa_consumo, mcuentas_saldo) as consumo_sobre_saldo
  , ratio_seguro(mpayroll, mrentabilidad) as payroll_sobre_rentabilidad
  , ratio_seguro(ctrx_quarter, cproductos) as trx_por_producto
from competencia_01
where foto_mes = 202104
limit 10

,numero_de_cliente,foto_mes,visa_utilizacion,master_utilizacion,consumo_sobre_caja,consumo_sobre_saldo,payroll_sobre_rentabilidad,trx_por_producto
0,12190227,202104,0.184176,0.019343,0.424482,0.080825,-174.315760,22.428571
1,12259566,202104,0.450946,NaN,0.446501,0.280794,0.000000,12.600000
2,12362153,202104,0.079061,0.113177,0.681790,-19.690430,-0.000000,29.200000
3,12371746,202104,0.707853,NaN,143.926755,13.826878,0.000000,5.166667
4,12389239,202104,0.510813,0.004921,1.920640,0.858001,0.000000,9.333333
5,12448223,202104,0.033683,0.043657,2.887531,0.100453,-0.000000,14.909091
6,12478582,202104,-0.001194,-0.000867,0.000000,-0.000000,0.000000,0.000000
7,12543441,202104,0.062554,0.002260,39.243119,-37.371598,0.000000,17.625000
8,12560418,202104,-0.064823,0.772560,0.601982,1.382918,0.000000,18.444444
9,12574444,202104,-0.000183,0.000000,0.000000,0.000000,-139.168246,8.428571


Y el control que conviene hacer siempre después de generar features: que no haya
quedado ningún `inf` ni `nan` suelto.

In [16]:
%%sql
with fe as (
    select
        ratio_seguro(Visa_msaldototal, Visa_mlimitecompra) as visa_utilizacion
      , ratio_seguro(Master_msaldototal, Master_mlimitecompra) as master_utilizacion
      , ratio_seguro(mtarjeta_visa_consumo, mcaja_ahorro) as consumo_sobre_caja
      , ratio_seguro(mtarjeta_visa_consumo, mcuentas_saldo) as consumo_sobre_saldo
      , ratio_seguro(mpayroll, mrentabilidad) as payroll_sobre_rentabilidad
      , ratio_seguro(ctrx_quarter, cproductos) as trx_por_producto
    from competencia_01
)
select
    sum(isinf(visa_utilizacion))::int + sum(isinf(master_utilizacion))::int
  + sum(isinf(consumo_sobre_saldo))::int + sum(isinf(consumo_sobre_caja))::int + sum(isinf(payroll_sobre_rentabilidad))::int
  + sum(isinf(trx_por_producto))::int                                       as infinitos
  , sum(isnan(visa_utilizacion))::int + sum(isnan(master_utilizacion))::int
  + sum(isnan(consumo_sobre_saldo))::int + sum(isnan(consumo_sobre_caja))::int + sum(isnan(payroll_sobre_rentabilidad))::int
  + sum(isnan(trx_por_producto))::int                                       as nans
from fe

,infinitos,nans
0,0,0


---

## Resumen para compartir

```sql
CREATE OR REPLACE MACRO ratio_seguro(a, b) AS
    case
        when a is null or b is null then null   -- falta el dato: no lo invento
        when a = 0 and b = 0        then 0      -- nada sobre nada: sin actividad
        when b = 0                  then null   -- a/0 es indefinido, no infinito
        else a / nullif(b, 0)                   -- red de seguridad
    end;

CREATE OR REPLACE MACRO ratio_seguro_def(a, b, defecto) AS
    coalesce(ratio_seguro(a, b), defecto);
```

Tres cosas que aprendí armándola:

1. **DuckDB no tira error al dividir por cero: devuelve `inf` y `nan` en silencio.**
   El problema no es que se rompa, es que no se rompe y sigue viaje hasta el modelo.
2. **`divide()` parece la solución y no lo es**: devuelve `null` en la división por
   cero, pero es división entera — `7/2` da `3`.
3. **`0/0` y `a/0` no son el mismo caso.** El primero es un cliente sin actividad y
   `0` lo describe bien; el segundo no tiene respuesta numérica honesta.

Y la decisión que no es de SQL sino del problema: **no convertir `null` en `0`**. En
este dataset el `null` de `Visa_*` significa "no tiene tarjeta", y eso multiplica por
5,5 la probabilidad de baja. Taparlo con un cero borra la señal más limpia del EDA.